import the data 
create the fields
find the fields, hopefully formation related ones, that are tied to success 
which formation values are tied to success if any?

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

In [ ]:

df = pd.read_csv('')  # your assembled play-level dataframe

# --- 1. Split tuple columns into flat x/y features ---
tuple_cols = [
    'average_offense_location', 'average_skill_position_location',
    'average_receiver_location', 'average_backfield_location',
    'average_defense_location', 'average_secondary_location',
    'average_front_location'
]

for col in tuple_cols:
    df[f'{col}_x'] = df[col].apply(lambda t: t[0] if isinstance(t, (tuple, list)) else np.nan)
    df[f'{col}_y'] = df[col].apply(lambda t: t[1] if isinstance(t, (tuple, list)) else np.nan)

df = df.drop(columns=tuple_cols)

# --- 2. Define target and leakage columns to exclude ---
target_col = 'big_play'
leakage_cols = ['rec_yards', 'rush_yards']  # outcome-derived, exclude from X

# --- 3. Define categorical columns ---
categorical_cols = [
    'left_label', 'right_label', 'backfield_desc', 'backfield_desc_with_motion',
    'team_coverage_type', 'team_coverage_man_zone', 'team_second_coverage_type',
    'team_second_coverage_man_zone', 'team_coverage_shell', 'play_type', 'play_direction'
]

for col in categorical_cols:
    df[col] = df[col].astype('category')

# --- 4. Build X / y ---
drop_cols = leakage_cols + [target_col]
X = df.drop(columns=drop_cols)
y = df[target_col].astype(int)

# --- 5. Time-aware split (if week is available upstream; else fallback to random) ---
if 'week' in df.columns:
    train_mask = df['week'] <= df['week'].quantile(0.8)
    X_train, X_test = X[train_mask], X[~train_mask]
    y_train, y_test = y[train_mask], y[~train_mask]
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

# --- 6. Train XGBoost with native categorical support ---
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    enable_categorical=True,
    tree_method='hist',
    eval_metric='auc',
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),  # handle class imbalance
    random_state=42
)

model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

# --- 7. Evaluate ---
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

print("AUC:", roc_auc_score(y_test, y_pred_proba))
print(classification_report(y_test, y_pred))

# --- 8. Feature importance ---
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances.head(20))